# 08 — Memory & Performance
**References:** Ramalho *Fluent Python* Ch. 2 · Python docs: `gc`, `tracemalloc`, `cProfile` · *High Performance Python* (Gorelick & Ozsvald)

## Narrative thread
```
Memory model -> Reference counting + gc -> Profiling workflow -> Slots -> Memoryview -> Numpy interop
```

## 1. Python memory model

CPython uses **reference counting** as the primary memory management strategy. Every object has a `ob_refcnt` field. When it hits 0, the object is immediately deallocated.

**Cyclic garbage collector (`gc` module):** Reference counting cannot collect cycles (`a.ref = b; b.ref = a`). The cyclic GC periodically sweeps for unreachable cycles in three generations (0, 1, 2).

```
Generation 0: newly created objects (collected most frequently)
Generation 1: survived one GC pass
Generation 2: survived two GC passes (long-lived; collected least often)
```

In [ ]:
import gc
import sys
import tracemalloc
import weakref
import cProfile
import pstats
import io
import timeit
import array

# ── Reference counting and object size ───────────────────────────────────
print('=== Object sizes ===')
for obj in [None, True, 42, 3.14, 'hello', b'hello', [], {}, set(), (), object()]:
    print(f'  {type(obj).__name__:12} {repr(obj)[:20]:20} {sys.getsizeof(obj):4} bytes')

print()
print('=== List growth (over-allocation) ===')
lst = []
prev = sys.getsizeof(lst)
for i in range(17):
    lst.append(i)
    new = sys.getsizeof(lst)
    if new != prev:
        print(f'  len={len(lst):2}: size jumped from {prev} to {new} bytes (realloc)')
        prev = new

print()
# ── Cyclic reference and the GC ──────────────────────────────────────────
class Node:
    def __init__(self, val): self.val = val; self.next = None
    def __del__(self): pass  # just for observation

gc.disable()  # manual control for demonstration
a = Node(1); b = Node(2)
a.next = b; b.next = a  # cycle!
del a, b                # refcount can't collect; cycles remain
collected = gc.collect()
gc.enable()
print(f'GC collected {collected} objects in the cycle')

In [ ]:
# ── tracemalloc: find where memory is allocated ───────────────────────────
tracemalloc.start(10)  # keep 10 frames

# Some allocations
big_list  = [list(range(100)) for _ in range(1000)]
big_dict  = {i: str(i)*100 for i in range(5000)}

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')[:5]
print('=== Top memory allocations ===')
for stat in top_stats:
    print(f'  {stat}')

tracemalloc.stop()

del big_list, big_dict
print()

# ── cProfile: find where time is spent ───────────────────────────────────
def slow_function():
    return sum(x**2 for x in range(100_000))

def medium_function():
    return [str(i) for i in range(50_000)]

def main():
    slow_function()
    medium_function()
    medium_function()

buf = io.StringIO()
pr  = cProfile.Profile()
pr.enable(); main(); pr.disable()
ps  = pstats.Stats(pr, stream=buf).sort_stats('cumulative')
ps.print_stats(6)
output = buf.getvalue().split('\n')
for line in output[5:12]:
    if line.strip():
        print(line)

In [ ]:
# ── timeit: micro-benchmarks ──────────────────────────────────────────────
print('=== Micro-benchmarks ===')

# List vs generator sum
t_list = timeit.timeit('sum([x**2 for x in range(1000)])', number=10_000)
t_gen  = timeit.timeit('sum(x**2 for x in range(1000))',   number=10_000)
print(f'sum([x**2 ...]) list: {t_list*1000:.1f} ms')
print(f'sum(x**2 ...)   gen:  {t_gen*1000:.1f} ms')

# String concatenation vs join
t_plus = timeit.timeit('s=""; [s:=s+str(i) for i in range(100)]', number=5_000)
t_join = timeit.timeit('"".join(str(i) for i in range(100))',      number=5_000)
print(f'String += concat: {t_plus*1000:.1f} ms')
print(f'str.join():       {t_join*1000:.1f} ms')

# dict vs try/except for key access
d = {i: i for i in range(1000)}
t_get   = timeit.timeit('d.get(500, None)', globals={'d': d}, number=500_000)
t_try   = timeit.timeit('try:\n v=d[500]\nexcept KeyError:\n v=None', globals={'d': d}, number=500_000)
print(f'dict.get():   {t_get*1000:.1f} ms')
print(f'try/except:   {t_try*1000:.1f} ms')

print()
# ── array module: typed arrays (no Python object overhead) ────────────────
import numpy as np

n = 1_000_000
py_list = list(range(n))
arr     = array.array('l', range(n))     # typed C array
np_arr  = np.arange(n, dtype=np.int64)

print('=== Memory comparison (1M integers) ===')
print(f'  Python list:   {sys.getsizeof(py_list)/1024:.0f} KB (+ {n*28/1024:.0f} KB for int objects)')
print(f'  array.array:   {sys.getsizeof(arr)/1024:.0f} KB')
print(f'  numpy.ndarray: {np_arr.nbytes/1024:.0f} KB')

t_list = timeit.timeit('sum(py_list)', globals={'py_list': py_list}, number=100)
t_np   = timeit.timeit('np_arr.sum()', globals={'np_arr': np_arr}, number=100)
print(f'\nsum(list):       {t_list*1000:.1f} ms')
print(f'np_arr.sum():    {t_np*1000:.1f} ms  ({t_list/t_np:.0f}x faster)')

In [ ]:
# ── memoryview: zero-copy buffer protocol ─────────────────────────────────
data = bytearray(b'Hello, World! ' * 100_000)

# Slicing bytearray creates a COPY
t_copy  = timeit.timeit('data[1000:5000]', globals={'data': data}, number=10_000)

# memoryview slice is ZERO-COPY — just a view into the buffer
mv = memoryview(data)
t_mv    = timeit.timeit('mv[1000:5000]',  globals={'mv': mv},   number=10_000)

print(f'bytearray slice (copy):  {t_copy*1000:.2f} ms')
print(f'memoryview slice (view): {t_mv*1000:.2f} ms  ({t_copy/t_mv:.0f}x faster)')

# memoryview can reshape (useful for image/audio processing)
raw = bytearray(range(256))
mv2 = memoryview(raw).cast('B', shape=[16, 16])  # reshape to 16x16 matrix
print(f'\nmemoryview reshaped: {mv2.shape} — row 5: {list(mv2[5])}')

## 2. Optimization hierarchy

1. **Algorithm** — $O(n)$ beats $O(n^2)$ every time, no matter the constant
2. **Data structures** — right container (set for membership, deque for FIFO)
3. **Python optimizations** — `__slots__`, list comprehensions, `str.join`, `local = self.method`
4. **NumPy/SciPy** — vectorize over Python loops
5. **C extensions** — Cython, ctypes, cffi (covered in notebook 12)

## 3. Key takeaways

| Concept | Rule |
|---|---|
| Reference counting | Immediate deallocation; cycles need `gc` |
| `tracemalloc` | Shows WHERE memory is allocated |
| `cProfile` | Shows WHERE time is spent — profile before optimizing |
| `timeit` | Micro-benchmark specific expressions |
| `array.array` | Typed C array; 8x smaller than Python list of ints |
| `memoryview` | Zero-copy slice of buffer — critical for high-throughput I/O |
| `__slots__` | 30–50% memory saving; no `__dict__` |

**Next:** notebook 09 — the type system: annotations, protocols, generics, and runtime checks.